# Plum With Jax

I am particularly impressed by two features of Julia as a programming language: just-in-time compilation and multiple dispatch.
Just-in-time compilation is a technique for improving the performance of interpreted languages by compiling them to native machine code "just in time" when they are run based on the types of the arguments.
This performance boost is particularly important for scientific computing, where it is common to run the same code over and over again on different data.
I would not be able to use Python for my research without just-in-time compilation.

The value of multiple dispatch on the other hand is a bit more theoretical.
But I'd like to try it out in a serious project to see if it helps me write better code.
Multiple dispatch is a generalization of object-oriented programming that allows you to define functions that behave differently depending on the types of all of their arguments.
The functions have the same name, but different implementations are called depending on the types of the arguments passed to the function when it is called.
The thing I like about multiple dispatch is that it sort of unbundles the data from the functions that operate on it compared to object-oriented programming.
Instead of associating methods with data structures by defining them within the class definition, I can define methods on types after they are defined or even if they are defined in a different package that I don't control.
And I can associate methods with multiple types at once, achieving patterns like inheritance without actually inheriting from a class.

More generally, multiple dispatch seems to provide an elegant way to enact polymorphism compared to object-oriented approaches I've implemented in the past.
For example, I've had a tricky time implementing variants of a certain model of human memory search in Python that share a lot of code but differ in a few key ways.
I originally used a form of inheritance to implement the variants, but I found that I had to override a lot of methods in the base class to get the behavior I wanted, and eventually faced an explosion of classes and methods as I tried to implement more variants.
Class composition was a bit better, but my code became very verbose as I had to write a lot of boilerplate to delegate method calls to the composed classes.
And refactoring to enable support for new functionality was a pain because I had to change the code in multiple places.
Multiple dispatch doesn't require me to define a class hierarchy at all, and only requires type annotations to indicate which methods are associated with which types, all reducing the amount of boilerplate I need to write.
This should make my code easier to refactor and extend.

Neither just-in-time compilation nor multiple dispatch are built into Python in the way they are into Julia, but there are third-party packages that provide similar functionality.
For just-in-time compilation, I've used JAX, which provides a just-in-time compiler for Python and NumPy code and many other cool features.
For multiple dispatch, I'm looking to use Plum, which extends Python's type annotation system to support multiple dispatch.
At the moment, I'm worried that while JAX and Plum are both great packages, they may not be compatible with each other.
Compiled functions in JAX are not Python functions, so I'm not sure if Plum will be able to dispatch on them.
I'm also concerned about the problem of providing meaningful type annotations for functions that operate on JAX arrays or other JAX objects.
A library called chex seems designed for providing type annotations for JAX code, but I'm not sure if it will work comfortably with Plum.
Finally, the magic of multiple dispatch is that it flexibly dispatches functions based on argument types right at run time; I suspect that this will be difficult to reconcile with just-in-time compilation, which ideally only occurs the first time a function is called.
Here, we'll implement some simple examples to see if we can get JAX and Plum to play nicely together.
If the experiment is successful, I'll try to use JAX and Plum to implement my model of human memory search in a way that is more elegant than my previous attempts, and hopefully more performant as well.

## Initial Demo
`flax` is a library for specifying and training neural networks in JAX. I'm specifically interested in its utility `flax.struct` for defining custom classes that can be used with JAX transformations such as `jax.jit`, the transformation that applies just-in-time compilation to a function. First we'll test that different classes defined and extended with `flax.struct` are compatible with Plum's `dispatch` mechanism.

In [1]:
from plum import dispatch
from flax import struct

# define two types
@struct.dataclass
class A:
    x: int

@struct.dataclass
class B:
    x: int

# define two functions with the same name but different argument types
@dispatch
def f(item: A, quantity: float) -> A:
    return item.replace(x = item.x + quantity + 1)

@dispatch
def f(item: B, quantity: float) -> B:
    return item.replace(x = item.x + quantity + 2)

# call the same function twice, but with different types
# check that the correct result is returned
assert f(A(x=2), 2.0).x == 5
assert f(B(x=2), 2.0).x == 6

Next we'll try different ways of applying `jax.jit` to functions decorated by `plum.dispatch`. First, we'll try applying `jax.jit` to the function itself. Here, we learn that `jax.jit` cannot be applied to transform a function configured for multiple dispatch:

In [3]:
from jax import jit

@jit
@dispatch
def f2(item: A, quantity: float) -> A:
    return item.replace(x = item.x + quantity + 1)

f2(A(x=2), 2.0)

NotFoundLookupError: For function `f2`, `(A(x=Traced<ShapedArray(int32[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>), Traced<ShapedArray(float32[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>)` could not be resolved.

Next we'll try applying it to a client function that calls the function. As before, we'll test whether type dispatch is preserved.

In [4]:
from jax import jit

@jit
def f3(item: A, quantity: float):
    return f(item, quantity)

f3(A(x=2), 2.0)

NotFoundLookupError: For function `f`, `(A(x=Traced<ShapedArray(int32[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>), Traced<ShapedArray(float32[], weak_type=True)>with<DynamicJaxprTrace(level=1/0)>)` could not be resolved.

In both cases, the arguments are interpreted as having or containing elements of type `Tracedwith` instead of `int` or `float`. That's how JAX works. It compiles using abstract values, not concrete values, so that it can produce a single compiled function that works for all possible values of the arguments. And that apparently involves passing arguments of abstract types to the function. 

How do I get around this? One approach is to use more general type annotations. For example, instead of `int` or `float`, I could use `Any`. But that defeats the purpose of type annotations, especially for performing multiple dispatch.

The `jaxtyping` library seems to provide a solution. It is apparently designed to work with runtime typechecking platforms like the main dependencies of `plum.dispatch` and of course it works with JAX really well itself.

In [9]:
from jax import jit
from plum import dispatch
from jaxtyping import Scalar

# define two types
class A(struct.PyTreeNode):
    x: Scalar

class B(struct.PyTreeNode):
    x: Scalar

class C(struct.PyTreeNode):
    x: Scalar

# define two functions with the same name but different argument types
@jit
@dispatch
def f(item: A, quantity: Scalar) -> A:
    return item.replace(x = item.x + quantity + 1)

@jit
@dispatch
def f(item: B, quantity: Scalar) -> B:
    return item.replace(x = item.x + quantity + 2)

assert f(A(x=2), 2.0).x == 5
assert f(B(x=2), 2.0).x == 6

Now we check whether a client function will properly dispatch to the correct argument...

In [10]:
@jit
@dispatch
def f(item: A, quantity: Scalar) -> A:
    return item.replace(x = item.x + quantity + 1)

@jit
@dispatch
def f(item: B, quantity: Scalar) -> B:
    return item.replace(x = item.x + quantity + 2)

@jit
def client_f(item, quantity):
    return f(item, quantity)

assert client_f(A(x=2), 2.0).x == 5
assert client_f(B(x=2), 2.0).x == 6
assert client_f(C(x=2), 2.0).x == 4

NotFoundLookupError: For function `f`, `(C(x=Traced<ShapedArray(int32[], weak_type=True)>with<DynamicJaxprTrace(level=2/0)>), Traced<ShapedArray(float32[], weak_type=True)>with<DynamicJaxprTrace(level=2/0)>)` could not be resolved.

It seems to work! Fascinating. So now I need to use what I've learned to decide on a design pattern.